# 06 Full randomized simulation (compact)
Each run samples **world premise**, **background events**, **agent roster**, **relationships**, **orchestration**, and **hyperparameters**.

For a **guided tour of every subsystem** plus the same style of randomized run with a **`SIMULATION_CONFIG`** dict, open **`07_simulation_exemplar.ipynb`**.

- **Mock** (default): set `NB_RANDOM_SEED` for a fixed draw.
- **HF**: set `HF_TOKEN` in `.env`, then `NB_USE_HF=1`.

Optional: `pip install ipywidgets` silences tqdm widget warnings in some Jupyter setups.

In [ ]:
from pathlib import Path
import sys
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    src = base / "src"
    if (src / "agent_rpg").is_dir():
        if str(src) not in sys.path:
            sys.path.insert(0, str(src))
        break
from agent_rpg.repo_root import find_repo_root
ROOT = find_repo_root()
print("Repository root:", ROOT)


In [ ]:
import os
import random

from agent_rpg import build_random_scenario

try:
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
except ImportError:
    pass

SEED = int(os.environ.get("NB_RANDOM_SEED", random.randint(1, 10_000_000)))
USE_HF = os.environ.get("NB_USE_HF", "").lower() in ("1", "true", "yes")
NUM_AGENTS = int(os.environ.get("NB_NUM_AGENTS", "0")) or None
MAX_ROUNDS = int(os.environ.get("NB_MAX_ROUNDS", "0")) or None

scenario = build_random_scenario(seed=SEED, num_agents=NUM_AGENTS, max_rounds=MAX_ROUNDS)
scenario.orchestration.enable_thought_phase = False
for a in scenario.agents:
    a.max_new_tokens = min(a.max_new_tokens, 180)

print("SEED", SEED)
print("Title:", scenario.world.title)
print("Agents:", [a.display_name for a in scenario.agents])
print("Turn order:", scenario.orchestration.turn_order)
print("World events:", len(scenario.world.background_events))

In [ ]:
_lines = []
for ev in scenario.world.background_events:
    rt = ev.round_trigger
    _lines.append(f"- [{ev.id}] (from round {rt}) {ev.description}")
print("\n".join(_lines) if _lines else "(no scripted world events)")

In [ ]:
import json
import os
import random

from agent_rpg import SimulationEngine
from agent_rpg.backends.fake import FakeLLMBackend
from agent_rpg.backends.hf_inference import HuggingFaceInferenceBackend

if USE_HF and os.environ.get("HF_TOKEN"):
    backend = HuggingFaceInferenceBackend()
    print("Backend: HuggingFaceInferenceBackend")
else:
    if USE_HF and not os.environ.get("HF_TOKEN"):
        print("USE_HF requested but HF_TOKEN missing — falling back to mock.")

    first_id = scenario.agents[0].agent_id

    def fake_factory(i, msgs):
        sysm = (msgs[0].get("content") or "") if msgs else ""
        if "scene director" in sysm.lower():
            return json.dumps({"next_agent_id": first_id})
        sp = (scenario.orchestration.stop_phrase or "") or ""
        opts = [
            {"thought": "weigh options", "say": "I propose we compare accounts.", "directed_at": None},
            {"thought": "pressure rising", "say": "Name your price, plainly.", "directed_at": None},
            {"thought": "", "say": "Let us not waste the night on insults.", "directed_at": None},
        ]
        r = random.Random(SEED + i)
        row = dict(opts[r.randint(0, len(opts) - 1)])
        if sp and r.random() < 0.25:
            row = {"thought": "endgame", "say": sp, "directed_at": None}
        return json.dumps(row)

    backend = FakeLLMBackend(factory=fake_factory)
    print("Backend: FakeLLMBackend (set NB_USE_HF=1 and HF_TOKEN for live)")

run_id = f"nb06_rand_{SEED}"
engine = SimulationEngine(scenario)
out = engine.run(backend, output_dir=ROOT / "runs", run_id=run_id, seed=SEED)
print("Artifacts:", out)

In [ ]:
from agent_rpg import ReportBuilder

rb = ReportBuilder.from_jsonl(out / "events.jsonl")
report_path = out / "report.md"
rb.write_markdown(report_path, scenario=scenario)
full = rb.to_dict(scenario=scenario)
summary = full["summary"]
social = full["social_dynamics"]
print("Report:", report_path)
print("Messages:", summary["message_count"], "Errors:", summary["error_count"])
print("Gini(turns):", round(social["gini_turns"], 4))

In [ ]:
try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    tc = social["turn_counts"]
    axes[0].bar(list(tc.keys()), list(tc.values()))
    axes[0].set_title("Messages per agent")
    de = {k: v for k, v in social.get("directed_edges", {}).items() if v and "->" in k}
    if de:
        axes[1].bar(list(de.keys()), list(de.values()))
        axes[1].tick_params(axis="x", rotation=45)
        axes[1].set_title("Directed edges (counts)")
    else:
        axes[1].set_visible(False)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("Plot skipped:", e)